Below is the code implementing the Full Primal, Interim Allocation, DSIC Linear Program. Just run the following cell to initialize the code.

In [59]:
from gurobipy import Model, GRB, quicksum
import numpy as np


def Primal_Interim(V, f):

    # Step 1: Create a new model
    model = Model("maximize_function")

    # Example dimensions for variables and vectors
    num_i = len(V[0])  # Number of bidders i
    num_j = len(V[0][0])  # Number of items j

    # Step 2: Define decision variables for y and p
    x = model.addVars(len(V), num_i, num_j, vtype=GRB.CONTINUOUS, name="y")  # y_i(v) as a decision variable
    p = model.addVars(len(V), num_i, vtype=GRB.CONTINUOUS, name="p")  # p_i(v) as a decision variable

    # Step 3: Define the function f(v)
    # Step 4: Set the objective
    model.setObjective(
        (quicksum(quicksum(f[v][i] * quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v, i] for v in range(len(V)))
        for i in range(num_i))
    ), 
    GRB.MAXIMIZE
    )

    # Step 5: Add constraints
    # Example: constraints on y and p (adjust based on your problem)
    
   # model.addConstrs((x[v,i,j] == x[v,i,j+1] 
      #                for i in range(num_i-1) 
       #               for j in range(num_j-1) 
            #          for v in range(len(V))
            #          if V[v][i][j]-V[v][i+1][j] == V[v][i][j+1]- V[v][i+1][j+1])
        
   # ,"Symmetry")
    

    model.addConstrs(
      ((quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) - p[v,i] >= quicksum(x[v_p, i, j] * V[v][i][j] for j in range(num_j))- p[v_p, i])
       for v_p in range(len(V)) 
       for v in range(len(V)) 
       for not_i in range(num_i)
       for i in range(num_i)
       if not_i != i
       if np.all(V[v][not_i]==V[v_p][not_i])
       
        ), "DSIC")

    model.addConstrs((p[v, i] - quicksum(x[v, i, j] * V[v][i][j] for j in range(num_j)) <= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "IR")


    model.addConstrs((quicksum(quicksum(f[v][i] * x[v, i, j] for v in range(len(V))) for i in range(num_i)) <= 1
                      for j in range(num_j)
                 ), "item feasibility")

    
    model.addConstrs((quicksum(x[v, i, j] for j in range(num_j)) <= 1
                      for v in range(len(V))
                      for i in range(num_i)
                      
                 ), "UD bidder feasibility")



    model.addConstrs((x[v, i, j] >= 0
                      for v in range(len(V))
                      for j in range(num_j)
                      for i in range(num_i)
                 ), "nonneg_y")

    model.addConstrs((p[v, i] >= 0
                      for v in range(len(V))
                      for i in range(num_i)
                 ), "nonneg_p")

    # Step 6: Optimize the model
    model.optimize()

    # Step 7: Print results
    if model.status == GRB.OPTIMAL:
        print(f"Optimal objective value: {model.objVal}")
        for v in range(len(V)):
            for i in range(num_i):
                for j in range(num_j):
                    print(f"x[{v},{i},{j}] = {x[v, i, j].X}, p[{v},{i}] = {p[v, i].X}")
    else:
        return("No optimal solution found.")
    return

Below are the parameters that we will enter into our linear program. They are currently set to have the type space be uniform [1,2] for a two bidder two item scenario. V contains the valuations for the bidders. f is the distribution of each v in V. f should sum to 1. It currently represents a uniform distribution. To solve the linear program with these parameters, just run the cell below. Reading the output of the program, the Optimal objective value will display the optimal output from the parameters. The display will then show each valuation and the allocation they receive. For example, y[3,0,1] = 1.0 means that in the fourth v (we start counting from 0), the first bidder (bidder 0) has an allocation of 1.0 for the second item (item 1).

In [60]:
V=[[[1,2],[1,2]],
   [[1,2],[1,1]],
   [[1,2],[2,2]],
   [[1,2],[2,1]],
  [[1,1],[1,2]],
  [[1,1],[1,1]],
  [[1,1],[2,2]],
  [[1,1],[2,1]],
   [[2,2],[1,2]],
   [[2,2],[1,1]],
   [[2,2],[2,2]],
   [[2,2],[2,1]],
  [[2,1],[1,2]],
  [[2,1],[1,1]],
  [[2,1],[2,2]],
  [[2,1],[2,1]]] # v matrices
#f = [1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16,1/16]  #distribution for which v is being chosen

f = [[1/16, 1/16] for _ in range(16)]


Primal_Interim(V,f)

Gurobi Optimizer version 11.0.3 build v11.0.3rc0 (mac64[x86] - Darwin 23.6.0 23G93)

CPU model: Intel(R) Core(TM) i5-1038NG7 CPU @ 2.00GHz
Thread count: 4 physical cores, 8 logical processors, using up to 8 threads

Optimize a model with 290 rows, 96 columns and 896 nonzeros
Model fingerprint: 0xfc299f6d
Coefficient statistics:
  Matrix range     [6e-02, 2e+00]
  Objective range  [6e-02, 1e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 1e+00]
Presolve removed 128 rows and 0 columns
Presolve time: 0.02s
Presolved: 162 rows, 96 columns, 800 nonzeros

Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    4.0000000e+00   1.399960e+02   0.000000e+00      0s
      56    3.5000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 56 iterations and 0.03 seconds (0.00 work units)
Optimal objective  3.500000000e+00
Optimal objective value: 3.5
x[0,0,0] = 0.0, p[0,0] = 0.0
x[0,0,1] = 1.0, p[0,0] = 0.0
x[0,1,0] = 0.0, p[0,1] = 0.0
x[0,1,1] = 1.0, p[0,1] 